# 02 — Pipeline tiền xử lý ảnh fundus

**Cell 5 → Cell 7**: kiểm tra trực quan từng bước (ROI → CLAHE → mặt nạ tròn → resize),
sau đó xử lý hàng loạt ra `data/processed/` và cập nhật lại đường dẫn trong split CSV.

Mục tiêu: ảnh đầu ra là **384×384×3**, vùng võng mạc chiếm tối đa khung hình,
không còn viền đen lớn.

## Cell 5 — Thử hàm ROI crop trên 1 ảnh

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").is_dir() and (PROJECT_ROOT.parent / "src").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.utils.config import load_project_configs
from src.preprocessing import (
    apply_clahe,
    apply_circular_mask,
    crop_black_border,
    preprocess_image,
)

configs = load_project_configs()
data_cfg = configs["data"]
pre_cfg = data_cfg["preprocessing"]

train_df = pd.read_csv(PROJECT_ROOT / "data/splits/train.csv")
row = train_df.iloc[0]

path = Path(str(row["image_path"]))
if not path.is_absolute():
    path = PROJECT_ROOT / "data/raw" / path

raw_bgr = cv2.imread(str(path), cv2.IMREAD_COLOR)
assert raw_bgr is not None, f"Không đọc được ảnh: {path}"
raw = cv2.cvtColor(raw_bgr, cv2.COLOR_BGR2RGB)

cropped = crop_black_border(
    raw,
    threshold=pre_cfg["roi"]["threshold"],
    margin=pre_cfg["roi"]["margin"],
    min_area_ratio=pre_cfg["roi"]["min_area_ratio"],
)

print("Ảnh gốc:", raw.shape, "-> sau ROI:", cropped.shape)
reduction = 1 - (cropped.shape[0] * cropped.shape[1]) / (raw.shape[0] * raw.shape[1])
print(f"Diện tích khung đen đã loại bỏ: {reduction:.1%}")

## Cell 6 — CLAHE (LAB) và mặt nạ tròn

- **CLAHE** chỉ tác động kênh L trong LAB → tăng tương phản cục bộ mà không lệch màu.
- **Mặt nạ tròn** xoá 4 góc đen để CNN không học artefact khung ảnh.

In [ ]:
enhanced = apply_clahe(
    cropped,
    clip_limit=pre_cfg["clahe"]["clip_limit"],
    tile_grid_size=tuple(pre_cfg["clahe"]["tile_grid_size"]),
)
masked = apply_circular_mask(
    enhanced, radius_ratio=pre_cfg["circular_mask"]["radius_ratio"]
)
final = preprocess_image(
    raw, image_size=data_cfg["image"]["size"], config=data_cfg
)

fig, axes = plt.subplots(1, 4, figsize=(18, 5))
for axis, image, title in zip(
    axes,
    [raw, cropped, masked, final],
    [
        f"Gốc {raw.shape[:2]}",
        f"ROI crop {cropped.shape[:2]}",
        f"CLAHE + mask {masked.shape[:2]}",
        f"Final {final.shape[:2]}",
    ],
):
    axis.imshow(image)
    axis.set_title(title, fontsize=10)
    axis.axis("off")
plt.tight_layout()
plt.show()

assert final.shape == (data_cfg["image"]["size"], data_cfg["image"]["size"], 3)
print("OK: ảnh cuối cùng", final.shape, final.dtype)

## Cell 7 — Xử lý hàng loạt ra `data/processed/`

Chạy 1 lần cho toàn bộ 16.266 ảnh (có thể mất 10–20 phút). Sau đó 3 file split
được cập nhật sang đường dẫn mới, và `DRDataset` sẽ đọc trực tiếp ảnh đã xử lý.

> Có thể bỏ qua bước này và huấn luyện trực tiếp trên ảnh gốc bằng cách truyền
> `preprocess=...` vào `DRDataset` (xử lý online, chậm hơn nhưng không tốn ổ đĩa).

In [ ]:
from tqdm.auto import tqdm

from src.data import load_splits, save_splits

raw_dir = PROJECT_ROOT / data_cfg["paths"]["raw_dir"]
processed_dir = PROJECT_ROOT / data_cfg["paths"]["processed_dir"]
processed_dir.mkdir(parents=True, exist_ok=True)

splits = load_splits(PROJECT_ROOT / data_cfg["paths"]["splits_dir"])
image_size = int(data_cfg["image"]["size"])
updated = {}
cache = {}

for split_name, frame in splits.items():
    frame = frame.copy()
    new_paths = []
    for path_value in tqdm(frame["image_path"], desc=split_name):
        source = Path(str(path_value))
        if not source.is_absolute():
            source = raw_dir / source

        if str(source) in cache:
            new_paths.append(cache[str(source)])
            continue

        image = cv2.imread(str(source), cv2.IMREAD_COLOR)
        if image is None:
            raise FileNotFoundError(source)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        processed = preprocess_image(image, image_size=image_size, config=data_cfg)
        relative = source.with_suffix(".png").name
        destination = processed_dir / relative
        if not destination.is_file():
            cv2.imwrite(str(destination), cv2.cvtColor(processed, cv2.COLOR_RGB2BGR))

        cache[str(source)] = str(destination.relative_to(PROJECT_ROOT))
        new_paths.append(cache[str(source)])

    frame["image_path"] = new_paths
    updated[split_name] = frame

save_splits(updated, out_dir=PROJECT_ROOT / data_cfg["paths"]["splits_dir"])
print(f"Đã xử lý {len(cache):,} ảnh -> {processed_dir}")